# 06 — Model Experiments

**IEEE-CIS Fraud Detection**

In this notebook we run more advanced fraud-model experiments. 
Since our baseline (notebook 05) showed poor PR-AUC and very low precision on linear models:

1. **Primary Model:** Gradient Boosted Trees (XGBoost)
   - Natively handles `NaN` (no imputing required).
   - Natively models non-linear relationships.
2. **Handle class imbalance natively:** We compute `scale_pos_weight = N_neg / N_pos` 
   from the **Training Set only**.
3. **Metrics:** Target metric is **PR-AUC**. We also track **ROC-AUC**, 
   **Precision, Recall**, and **F1**.
4. **Threshold optimisation:** The default `0.5` threshold is arbitrary. We search 
   for the threshold that maximises the **F1 score** on the validation set.

> **Status:** Test set is completely excluded here to avoid leakage.

---
## 1 · Imports & Setup

In [ ]:
import sys
from pathlib import Path
import copy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb

from sklearn.metrics import (
    roc_auc_score, average_precision_score, precision_recall_curve,
    precision_score, recall_score, f1_score
)

PROJECT_ROOT = Path.cwd().parent
PROC_DIR     = PROJECT_ROOT / "data" / "processed"
METRICS_DIR  = PROJECT_ROOT / "reports" / "metrics"
FIG_DIR      = PROJECT_ROOT / "reports" / "figures"
MODELS_DIR   = PROJECT_ROOT / "models"

for d in [METRICS_DIR, FIG_DIR, MODELS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 60)
sns.set_theme(style="whitegrid", palette="muted")
%matplotlib inline

TARGET = "isFraud"

---
## 2 · Load Pre-processed Splits

In [ ]:
X_train = pd.read_parquet(PROC_DIR / "X_train.parquet")
X_valid = pd.read_parquet(PROC_DIR / "X_valid.parquet")
y_train = pd.read_parquet(PROC_DIR / "y_train.parquet")[TARGET]
y_valid = pd.read_parquet(PROC_DIR / "y_valid.parquet")[TARGET]

print(f"Train       : {X_train.shape[0]:>8,} rows,  y mean {y_train.mean():>6.2%}")
print(f"Validation  : {X_valid.shape[0]:>8,} rows,  y mean {y_valid.mean():>6.2%}")

In [ ]:
# Ensure only numeric inputs for XGBoost — categorical encoding should have been done.
# If any object columns remain, we drop them here:
obj_cols_train = X_train.select_dtypes(["object", "category"]).columns
if len(obj_cols_train) > 0:
    print(f"Dropping categorical cols (should be sparse-encoded in pipeline): {list(obj_cols_train)}")
    X_train = X_train.drop(columns=obj_cols_train)
    X_valid = X_valid.drop(columns=obj_cols_train)

---
## 3 · Scale Pos Weight Computation

XGBoost parameters allow adjusting base weights for positive samples directly. 
`scale_pos_weight = Count(Negative) / Count(Positive)`

In [ ]:
pos_count = y_train.sum()
neg_count = len(y_train) - pos_count
spw       = float(neg_count / pos_count)

print(f"Training set Negatives : {neg_count:,}")
print(f"Training set Positives : {pos_count:,}")
print(f"scale_pos_weight       : {spw:.2f}")

---
## 4 · Train XGBoost with Early Stopping

We use **PR-AUC** as the metric for early stopping (since accuracy and ROC-AUC can be misleading).

In [ ]:
%%time

params = {
    "objective": "binary:logistic",
    "eval_metric": "aucpr",    # Optimize PR-AUC for early stopping
    "scale_pos_weight": spw,   # Handle severe imbalance
    "eta": 0.05,
    "max_depth": 6,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "tree_method": "hist",    # Much faster for large datasets
    "random_state": 42,
}

# Create XGBoost DMatrix objects
dtrain = xgb.DMatrix(X_train, label=y_train)
dvalid = xgb.DMatrix(X_valid, label=y_valid)

evals = [(dtrain, "train"), (dvalid, "valid")]

print("Training XGBoost...")
model = xgb.train(
    params=params,
    dtrain=dtrain,
    num_boost_round=1000,           # Generous rounds
    evals=evals,
    early_stopping_rounds=50,       # Stop if PR-AUC stalls for 50 trees
    verbose_eval=50,
)

print(f"\nBest Iteration: {model.best_iteration}")
print(f"Best PR-AUC   : {model.best_score:.4f}")

---
## 5 · Predict & Evaluate (Validation Split)

In [ ]:
y_prob = model.predict(dvalid)

roc_auc = roc_auc_score(y_valid, y_prob)
pr_auc  = average_precision_score(y_valid, y_prob)

print("Validation Scores (Raw Probabilities):")
print(f"ROC-AUC: {roc_auc:.4f}")
print(f"PR-AUC : {pr_auc:.4f}")

---
## 6 · Threshold Optimization

We search the PR Curve for the threshold that yields the best **F1-Score**.
In practice, a business might prefer more precision or more recall.

In [ ]:
precisions, recalls, thresholds = precision_recall_curve(y_valid, y_prob)

# F1 calculation (handling divisions by zero if precision+recall=0)
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-9)

# Locate best threshold based on F1
best_idx       = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]
best_f1        = f1_scores[best_idx]

print(f"Optimised Threshold: {best_threshold:.4f}")
print(f"Maximum F1 Score   : {best_f1:.4f}")

In [ ]:
y_pred_opt = (y_prob >= best_threshold).astype(int)

eval_dict = {
    "ROC-AUC": roc_auc,
    "PR-AUC" : pr_auc,
    "F1"     : f1_score(y_valid, y_pred_opt),
    "Recall" : recall_score(y_valid, y_pred_opt),
    "Precision": precision_score(y_valid, y_pred_opt),
}

print("--- Metric Profile with Optimal Threshold ---")
for k, v in eval_dict.items():
    print(f"{k:<12}: {v:.4f}")

print("—" * 30)
y_pred_05 = (y_prob >= 0.5).astype(int)
print(f"Default (0.5) F1: {f1_score(y_valid, y_pred_05):.4f} |")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(pd.crosstab(y_valid, y_pred_opt, rownames=["Actual"], colnames=["Predicted"]),
            annot=True, fmt='d', cmap="Blues", cbar=False, annot_kws={"fontsize": 14})
ax.set_title(f"Confusion Matrix (Threshold = {best_threshold:.3f})")
fig.savefig(FIG_DIR / "fig_xgb_valid_confusion.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(recalls, precisions, label="XGBoost PR Curve", color="#DD4A48", lw=2)
# Mark optimal threshold
ax.scatter(recalls[best_idx], precisions[best_idx], color="blue", s=100, 
           label=f"Optimal F1 (Thresh={best_threshold:.3f})")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Precision-Recall Curve (Validation Set)")
ax.legend()
fig.savefig(FIG_DIR / "fig_xgb_pr_curve.png", dpi=150, bbox_inches="tight")
plt.show()

> **Observation:** The optimal threshold balances precision and recall safely.  
> This clearly outperforms the baselines due to missing value handling,  
> `scale_pos_weight`, and trees naturally detecting non-linear interactions.

---
## 7 · Feature Importance Overview

Plot built-in XGBoost weight importance. (For deep dive, we'll use SHAP in notebook 08).

In [ ]:
importance = pd.Series(model.get_score(importance_type="gain")).sort_values(ascending=False)
top20 = importance.head(20)

fig, ax = plt.subplots(figsize=(10, 8))
sns.barplot(x=top20.values, y=top20.index, palette="viridis", ax=ax)
ax.set_title("XGBoost Top 20 Features (Gain)")
ax.set_xlabel("Average Gain across splits")
fig.savefig(FIG_DIR / "fig_xgb_feat_importance.png", dpi=150, bbox_inches="tight")
plt.show()

> **Observation:** We can see which aggregated features or original columns carry 
> the most predictive power. Often `TransactionAmt`, D columns, and `card` aggregations dominate.

---
## 8 · Save Model & Threshold

Save the best iteration to disk to use against the frozen test set in Notebook 07.

In [ ]:
# Save XGB model using native .json or generic wrapper
model_path = MODELS_DIR / "xgb_model_validateset.json"
model.save_model(model_path)

# Save optimal threshold & metrics
summary = {
    "threshold": float(best_threshold),
    "metrics": {
        k: float(v) for k, v in eval_dict.items()
    }
}
with open(METRICS_DIR / "xgb_experiment_metrics.json", "w") as f:
    json.dump(summary, f, indent=4)

print(f"Saved Model     → {model_path}")
print(f"Saved Threshold → {METRICS_DIR / 'xgb_experiment_metrics.json'}")

### Conclusion

- XGBoost successfully models the imbalanced problem with `scale_pos_weight`.
- `PR-AUC` tracking prevents the model from looking blindly good at accuracy.
- Setting the optimal F1 threshold yields a tangible, viable prediction array.
- All features in this experiment were computed securely and chronologically pre-split.

### Next Notebook
→ **`07_final_evaluation.ipynb`** — Evaluating the chosen (saved) model on the strictly held-out test set.

In [ ]:
print("Notebook 06 complete. ✓")